# Excel Inspection and Export Workflow

This notebook inspects the yearly poker statistics workbook, explores the data, and exports a consolidated dataset for analysis and visualization.

## 1. Install and Import Libraries

Install required packages if needed, then import the necessary libraries.

In [ ]:
# If pandas or openpyxl are not installed, uncomment the next line:
# !pip install pandas openpyxl matplotlib seaborn

import pathlib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8')

## 2. Load Sample Data

Open the Excel workbook, inspect sheet names, and load a sample of each sheet.

In [ ]:
excel_path = pathlib.Path('LDVP - Statistiques annuelles.xlsx')
assert excel_path.exists(), f'Workbook not found: {excel_path}'

workbook = pd.ExcelFile(excel_path)
sheet_names = workbook.sheet_names
print('Sheets found:', sheet_names)

sheets = {name: workbook.parse(name) for name in sheet_names}
for name, df in sheets.items():
    print(f'--- Sheet: {name} | rows={len(df)} cols={len(df.columns)} ---')
    display(df.head(3))

## 3. Explore the Dataset

Inspect data structure, summary statistics, and missing values.

In [ ]:
sheet = sheet_names[0]  # start with the first sheet
df = sheets[sheet].copy()

print('Columns:', list(df.columns))
print('Data types:')
print(df.dtypes)

print('Missing values by column:')
print(df.isna().sum())

display(df.describe(include='all').T)

## 4. Visualize Key Features

Create simple visualizations for common poker statistics, such as result distributions and trends.

In [ ]:
def plot_if_column_exists(df, col, kind='hist', title=None, **kwargs):
    if col in df.columns:
        title = title or f'{col} distribution'
        fig, ax = plt.subplots(figsize=(8, 4))
        if kind == 'hist':
            sns.histplot(df[col].dropna(), kde=True, ax=ax, **kwargs)
        elif kind == 'line':
            df[col].dropna().reset_index(drop=True).plot(ax=ax)
        ax.set_title(title)
        plt.show()
    else:
        print(f'Column not found: {col}')

plot_if_column_exists(df, 'Net', kind='hist')
plot_if_column_exists(df, 'Profit', kind='hist')
plot_if_column_exists(df, 'Date', kind='line')

## 5. Perform Basic Analysis

Compute aggregated metrics and derive insights from the dataset.

In [ ]:
def try_parse_date(series):
    try:
        return pd.to_datetime(series, errors='coerce')
    except Exception:
        return pd.Series([pd.NaT] * len(series))

for date_col in ['Date', 'date', 'DATE']:
    if date_col in df.columns:
        parsed = try_parse_date(df[date_col])
        if parsed.notna().any():
            df['__parsed_date'] = parsed
            break

if '__parsed_date' in df.columns:
    df['Year'] = df['__parsed_date'].dt.year
    print('Year distribution:')
    print(df['Year'].value_counts().sort_index())

for value_col in ['Profit', 'Net', 'Gain', 'Résultat', 'Result']:
    if value_col in df.columns:
        print(f'
Summary for {value_col}:')
        display(df[value_col].describe())
        if 'Year' in df.columns:
            display(df.groupby('Year')[value_col].sum().reset_index().sort_values('Year'))
        break

## 6. Save and Export Results

Export the consolidated dataset for future analysis and visualization.

In [ ]:
common_columns = set.intersection(*(set(x.columns) for x in sheets.values()))
if len(sheets) > 1 and common_columns:
    dfs = [sheets[name][sorted(common_columns)] for name in sheet_names]
    combined = pd.concat(dfs, ignore_index=True)
else:
    combined = df.copy()

combined.to_csv('sessions.csv', index=False, encoding='utf-8-sig')
print('Exported consolidated data to sessions.csv')

for name, sheet_df in sheets.items():
    sheet_df.to_csv(f'{name}.csv', index=False, encoding='utf-8-sig')
print('Exported individual sheet CSV files')